## Sezione 1 — Setup & Imports

Carichiamo le dipendenze, leggiamo la chiave API Groq dal file `.env` (radice del progetto), istanziamo il client `ChatGroq` (modello Llama 3.3 70B su infrastruttura Groq) e verifichiamo con una piccola call di handshake che la chiave sia valida e il modello risponda.

In [ ]:
import os, json
import pandas as pd
import numpy as np
from typing import TypedDict, List, Optional, Annotated, Dict, Any
import operator
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

# Load .env from project root (notebook lives in agents/, .env is one level up)
_ENV = Path.cwd() / '.env'
if not _ENV.exists():
    _ENV = Path.cwd().parent / '.env'
load_dotenv(dotenv_path=_ENV)

# Groq exposes a fast OpenAI-compatible inference for Llama 3 family models.
# Set GROQ_API_KEY in .env (gsk_...).
GROQ_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_KEY:
    raise RuntimeError(
        "GROQ_API_KEY non trovata in .env. Aggiungi una riga GROQ_API_KEY=gsk_... al file .env."
    )

GROQ_MODEL = os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile")
llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
    api_key=GROQ_KEY,
    timeout=60,
)
_provider = f"{GROQ_MODEL} (Groq)"

# Verifica operativa — fallisce esplicitamente se la chiave non è valida
response = llm.invoke("Reply with exactly one word: OK")
assert "ok" in response.content.lower(), f"Risposta inattesa: {response.content}"
print(f"LLM API: OK — provider: {_provider}")

## Phase 2 — Data Exploration

Before building any agent or quality tool, we must understand the raw state of each dataset. 
This phase profiles all four CSVs along four dimensions: **schema structure**, **completeness**, 
**format consistency**, and **disguised null patterns**. The anomalies discovered here directly 
motivate every deterministic tool built in Phase 3.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

spesa = pd.read_csv("data/project_data_quality/spesa.csv")
attivazioni = pd.read_csv("data/project_data_quality/attivazioniCessazioni.csv")
tipologia = pd.read_csv("data/project_anomaly_detection/TIPOLOGIA_VIAGGIATORE.csv")
allarmi = pd.read_csv("data/project_anomaly_detection/ALLARMI.csv")

datasets = {
    "spesa": spesa,
    "attivazioni": attivazioni,
    "tipologia": tipologia,
    "allarmi": allarmi
}

for name, df in datasets.items():
    print(f"[{name}]  rows={df.shape[0]}  columns={df.shape[1]}")

### 2.1 — Schema Audit: Column Naming Conventions

We inspect all column names for violations of standard naming conventions. 
A well-formed schema uses lowercase snake_case, avoids special characters, 
and never starts with a digit. Violations here will break downstream 
SQL ingestion, pandas attribute access, and API serialization.

In [ ]:
import re
import pandas as pd

def audit_schema(df, name):
    issues = []
    for col in df.columns:
        flags = []
        if col != col.lower():
            flags.append("mixed_case")
        if re.search(r'[%\s-]', col):  # hyphen at end of class
            flags.append("special_char_or_space")
        if re.match(r'^\d', col):
            flags.append("starts_with_digit")
        if flags:
            issues.append({"dataset": name, "column": col, "violations": ", ".join(flags)})
    return pd.DataFrame(issues)

results = [audit_schema(df, name) for name, df in datasets.items()]
schema_issues = pd.concat(results, ignore_index=True) if results else pd.DataFrame()

if schema_issues.empty:
    print("No schema issues found.")
else:
    print(schema_issues.to_string(index=False))

# completeness profile

In [ ]:
def completeness_profile(df, name):
    DISGUISED_NULLS = {"n.d.", "nd", "n/a", "null", "none", "?", "-", "//", 
                       "unknown", "na", "missing", "", " "}
    
    result = []
    for col in df.columns:
        total = len(df)
        real_nulls = df[col].isna().sum()
        
        str_col = df[col].astype(str).str.strip().str.lower()
        disguised = str_col.isin(DISGUISED_NULLS).sum()
        
        effective_missing = real_nulls + disguised
        completeness_pct = round(100 * (1 - effective_missing / total), 2)
        
        result.append({
            "dataset": name, "column": col,
            "null_count": real_nulls,
            "disguised_null_count": disguised,
            "completeness_%": completeness_pct
        })
    return pd.DataFrame(result)

completeness = pd.concat([completeness_profile(df, name) for name, df in datasets.items()])

# Show the most problematic columns
sparse = completeness[completeness["completeness_%"] < 85].sort_values("completeness_%")
print(sparse[["dataset","column","null_count","disguised_null_count","completeness_%"]].to_string(index=False))

# format consistency detection

In [ ]:
# Focus on date/temporal columns and key categorical columns

def sample_unique_values(df, col, n=8):
    return df[col].dropna().astype(str).str.strip().unique()[:n].tolist()

# --- DATE FORMAT AUDIT ---
date_cols = {
    "spesa":       ["rata", "aggregation-time"],
    "attivazioni": ["mese", "anno", "RATA", "aggregation-time"],
    "tipologia":   ["DATA_PARTENZA", "ANNO_PARTENZA"],
    "allarmi":     ["DATA_PARTENZA", "ANNO_PARTENZA"]
}

for ds_name, cols in date_cols.items():
    df = datasets[ds_name]
    print(f"\n=== {ds_name.upper()} ===")
    for col in cols:
        if col in df.columns:
            samples = sample_unique_values(df, col)
            print(f"  [{col}]: {samples}")

# outliers and categorical anomaly scan

In [ ]:
# Numerical outlier hints + impossible values
print("=== NUMERICAL ANOMALIES ===")

# spesa: spesa column with currency symbol
spesa_currency = spesa[spesa['spesa'].astype(str).str.contains('€', na=False)]
print(f"[spesa] Rows with '€' in `spesa` column: {len(spesa_currency)}")

# allarmi: negative TOT value
allarmi_neg = allarmi[pd.to_numeric(allarmi['TOT'], errors='coerce') < 0]
print(f"[allarmi] Rows with negative TOT: {len(allarmi_neg)}")

# allarmi: TOT as non-numeric string
allarmi_bad_tot = allarmi[pd.to_numeric(allarmi['TOT'], errors='coerce').isna() & allarmi['TOT'].notna()]
print(f"[allarmi] Rows with non-numeric TOT (e.g. '~1', 'N.D.', '1 voli'): {len(allarmi_bad_tot)}")
print(f"  Samples: {allarmi_bad_tot['TOT'].unique()[:6].tolist()}")

# attivazioni: extreme attivazioni/cessazioni values
att_col = pd.to_numeric(attivazioni['attivazioni'], errors='coerce')
print(f"\n[attivazioni] `attivazioni` — max={att_col.max()}, 99th pct={att_col.quantile(0.99):.0f}")
ces_col = pd.to_numeric(attivazioni['cessazioni'], errors='coerce')
print(f"[attivazioni] `cessazioni` — max={ces_col.max()}, 99th pct={ces_col.quantile(0.99):.0f}")

# tipologia: INVESTIGATI > ENTRATI (logical impossibility)
tip = tipologia.copy()
tip['ENTRATI_n']    = pd.to_numeric(tip['ENTRATI'],    errors='coerce')
tip['INVESTIGATI_n']= pd.to_numeric(tip['INVESTIGATI'],errors='coerce')
cross_issue = tip[tip['INVESTIGATI_n'] > tip['ENTRATI_n']]
print(f"\n[tipologia] Rows where INVESTIGATI > ENTRATI (logical violation): {len(cross_issue)}")

## Phase 3 — Deterministic Tool Library

In this phase we convert the exploratory findings from Phase 2 into a deterministic validation layer.  
Each function acts like a strict audit tool: it returns structured evidence, severity, and remediation hints instead of free-form text.

These tools will later be exposed to the LangGraph agents so the LLM can reason on top of hard evidence rather than guessing from raw CSV contents.

In [ ]:
import re
import json
import numpy as np
import pandas as pd
from collections import defaultdict

DISGUISED_NULLS = {
    "n.d.", "nd", "n/a", "null", "none", "?", "-", "//",
    "unknown", "na", "missing", "", " "
}

SEVERITY_RANK = {"low": 1, "medium": 2, "high": 3, "critical": 4}

# Strips surrounding whitespace and returns None for empty strings.
def normalize_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x != "" else None

# Lower-cases a value after normalizing it; used when comparing tokens case-insensitively.
def normalize_token(x):
    x = normalize_text(x)
    return x.lower() if x is not None else None

# Returns True when a value is semantically empty (placeholder string) even if pandas sees it as non-null.
def is_disguised_null(x):
    token = normalize_token(x)
    return token in DISGUISED_NULLS if token is not None else False

# Pulls up to n distinct non-null string samples from a series for use as evidence in issue reports.
def safe_samples(series, n=5):
    s = series.dropna().astype(str).str.strip()
    s = s[s.ne("")]
    return s.drop_duplicates().head(n).tolist()

# Attempts a best-effort numeric parse by stripping currency symbols, spaces,
# thousand-dot separators, and comma decimals before calling pd.to_numeric.
def coerce_numeric_loose(series):
    s = series.astype(str).str.strip()
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan, "none": np.nan})
    s = s.str.replace("€", "", regex=False)
    s = s.str.replace(r"\s+", "", regex=True)
    s = s.str.replace(r"(?<=\d)\.(?=\d{3}(\D|$))", "", regex=True)
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9.\-]", "", regex=True)
    s = s.replace({"": np.nan, "-": np.nan, ".": np.nan, "-.": np.nan})
    return pd.to_numeric(s, errors="coerce")

# Guesses whether a column is "numeric", "date", or "string" based on
# what fraction of present values can be parsed as each type.
def infer_semantic_type(series, threshold=0.90):
    s = series.copy()
    mask_present = s.notna() & s.astype(str).str.strip().ne("")
    s = s[mask_present]
    if len(s) == 0:
        return "unknown"
    numeric_rate = coerce_numeric_loose(s).notna().mean()
    date_rate = pd.to_datetime(s.astype(str).str.strip(), errors="coerce", dayfirst=True).notna().mean()
    if numeric_rate >= threshold:
        return "numeric"
    if date_rate >= threshold:
        return "date"
    return "string"

# Builds a single issue dict with a consistent schema so all tools speak the same language
# and LangGraph agents can parse results without any special-casing.
def make_issue(dataset, tool, issue_type, severity, message,
               columns=None, row_count=None, evidence=None, suggested_fix=None):
    return {
        "dataset": dataset,
        "tool": tool,
        "issue_type": issue_type,
        "severity": severity,
        "columns": columns if columns is not None else [],
        "row_count": int(row_count) if row_count is not None else None,
        "message": message,
        "evidence": evidence if evidence is not None else {},
        "suggested_fix": suggested_fix
    }

# Wraps a list of issues from one tool into a standard result envelope that includes
# an issue count, a per-severity breakdown, and optional metadata.
def build_result(dataset, tool, issues, meta=None):
    severity_breakdown = defaultdict(int)
    for issue in issues:
        severity_breakdown[issue["severity"]] += 1
    return {
        "dataset": dataset,
        "tool": tool,
        "issue_count": len(issues),
        "severity_breakdown": dict(severity_breakdown),
        "issues": issues,
        "meta": meta or {}
    }

## 3.1 — Schema Discovery & Validation (generic)

Lo schema atteso non è hardcoded per dataset specifici. La funzione `discover_dataset_rules(df, name)` ispeziona un sample del DataFrame e popola al volo `EXPECTED_SCHEMAS`, `MANDATORY_COLUMNS`, `FORMAT_RULES`, `NUMERIC_RULES`. Il risultato: la pipeline funziona su **qualsiasi CSV**, non solo sui 4 NoiPA usati in fase di test.

Il `check_schema` riusa il sistema di issue/severity standardizzato e produce: `missing_required_columns`, `duplicate_column_names`, `semantic_type_mismatch`, `naming_convention_violation`. La voce `unexpected_columns` non c'è più — non è significativa quando lo schema atteso è derivato dal df stesso.


In [ ]:
# ─── Generic schema discovery (no hardcoded per-dataset rules) ──────────────
# We DO NOT hardcode schemas for specific datasets. Rules are discovered from
# the dataframe sample at runtime, then cached per dataset_name.

EXPECTED_SCHEMAS = {}        # populated by discover_dataset_rules
SCHEMA_NAMING_RULES = {"snake_case_preferred": True}

def discover_dataset_rules(df, name, sample_size=300):
    """Discover schema/mandatory/format/numeric rules from the dataframe itself.
    Heuristic-only (no LLM call) to keep token cost zero for the discovery step.

    Side-effect: also populates MANDATORY_COLUMNS, FORMAT_RULES, NUMERIC_RULES,
    DUPLICATE_KEY_RULES if they exist as globals.
    """
    sample = df.sample(min(sample_size, len(df)), random_state=42) if len(df) > sample_size else df
    n = len(df)

    # Per-column inferred semantic type (numeric/date/string)
    expected_types = {}
    for col in df.columns:
        expected_types[col] = infer_semantic_type(sample[col])

    EXPECTED_SCHEMAS[name] = {
        "required_columns": list(df.columns),     # everything currently present is "expected"
        "expected_types": expected_types
    }

    # MANDATORY: columns with effective completeness >= 50% AND not flagged as sparse.
    # Rationale: a column that is mostly populated is treated as load-bearing.
    mandatory = []
    for col in df.columns:
        nulls = df[col].isna().sum()
        disguised = df[col].astype(str).str.strip().str.lower().isin(DISGUISED_NULLS).sum()
        completeness = 1 - (nulls + disguised) / n if n else 0
        if completeness >= 0.50:
            mandatory.append(col)
    if "MANDATORY_COLUMNS" in globals():
        MANDATORY_COLUMNS[name] = mandatory

    # FORMAT_RULES: any column whose semantic type is "date" gets format="date".
    if "FORMAT_RULES" in globals():
        FORMAT_RULES[name] = {col: "date" for col, t in expected_types.items() if t == "date"}

    # NUMERIC_RULES: any numeric column gets {min: 0} as a safe default
    # (PA datasets are typically counts/amounts, never negative). User can override
    # via the Streamlit advanced panel.
    if "NUMERIC_RULES" in globals():
        NUMERIC_RULES[name] = {col: {"min": 0} for col, t in expected_types.items() if t == "numeric"}

    # DUPLICATE_KEY_RULES: heuristic — pick high-cardinality columns as candidate keys.
    if "DUPLICATE_KEY_RULES" in globals():
        DUPLICATE_KEY_RULES[name] = []  # disabled by default; user opt-in

    return {
        "schema": EXPECTED_SCHEMAS[name],
        "mandatory": mandatory,
        "format_rules": {col: "date" for col, t in expected_types.items() if t == "date"},
        "numeric_rules": {col: {"min": 0} for col, t in expected_types.items() if t == "numeric"},
    }


# Validates a DataFrame's structure against the discovered/expected schema.
def check_schema(df, dataset_name, expected_schemas=None):
    expected_schemas = expected_schemas if expected_schemas is not None else EXPECTED_SCHEMAS
    tool = "check_schema"
    issues = []
    expected = expected_schemas.get(dataset_name, {})

    required_columns = expected.get("required_columns", [])
    expected_types = expected.get("expected_types", {})
    actual_columns = list(df.columns)

    missing_required = [c for c in required_columns if c not in actual_columns]
    duplicate_columns = df.columns[df.columns.duplicated()].tolist()

    if missing_required:
        issues.append(make_issue(
            dataset_name, tool, "missing_required_columns", "critical",
            f"Missing required columns: {missing_required}",
            columns=missing_required, row_count=len(df),
            evidence={"missing_required": missing_required},
            suggested_fix="Restore missing columns before downstream validation."
        ))

    if duplicate_columns:
        issues.append(make_issue(
            dataset_name, tool, "duplicate_column_names", "high",
            f"Duplicate column names detected: {duplicate_columns}",
            columns=duplicate_columns, row_count=len(df),
            evidence={"duplicate_columns": duplicate_columns},
            suggested_fix="Rename duplicate headers."
        ))

    actual_types = {col: infer_semantic_type(df[col]) for col in df.columns}
    for col, expected_type in expected_types.items():
        if col in df.columns:
            actual_type = actual_types[col]
            if expected_type != "unknown" and actual_type != "unknown" and actual_type != expected_type:
                issues.append(make_issue(
                    dataset_name, tool, "semantic_type_mismatch", "high",
                    f"Column `{col}` looks like {actual_type}, expected {expected_type}.",
                    columns=[col], row_count=int(df[col].notna().sum()),
                    evidence={"expected_type": expected_type, "actual_type": actual_type,
                              "samples": safe_samples(df[col])},
                    suggested_fix="Standardize the column representation."
                ))

    for col in df.columns:
        flags = []
        if SCHEMA_NAMING_RULES["snake_case_preferred"]:
            if re.search(r"[\s%]", col):
                flags.append("special_char_or_space")
        if col[:1].isdigit():
            flags.append("starts_with_digit")
        if flags:
            issues.append(make_issue(
                dataset_name, tool, "naming_convention_violation", "low",
                f"Column `{col}` violates preferred naming conventions.",
                columns=[col], row_count=len(df),
                evidence={"violations": flags},
                suggested_fix="Standardize column naming."
            ))

    return build_result(dataset_name, tool, issues, meta={"column_count": len(df.columns)})


## 3.2 — Completeness Tools

Questi tool quantificano la *missingness effettiva*, non solo i null pandas. Token come `N.D.`, `?`, `-` sono semanticamente vuoti anche quando tecnicamente non sono null. La severity sale a `critical` per le colonne mandatory con missing > 20%.


In [ ]:
MANDATORY_COLUMNS = {}  # populated by discover_dataset_rules at runtime

# Combines real nulls and disguised null tokens to compute effective missingness per column.
# Severity escalates to critical when mandatory columns exceed 20% effective missingness.
def check_nulls(df, dataset_name, mandatory_columns=MANDATORY_COLUMNS):
    tool = "check_nulls"
    issues = []
    mandatory = mandatory_columns.get(dataset_name, [])

    for col in df.columns:
        total = len(df)
        real_nulls = int(df[col].isna().sum())
        disguised_nulls = int(df[col].astype(str).str.strip().str.lower().isin(DISGUISED_NULLS).sum())
        effective_missing = real_nulls + disguised_nulls
        missing_ratio = effective_missing / total if total else 0

        if effective_missing == 0:
            continue

        if col in mandatory:
            severity = "critical" if missing_ratio >= 0.20 else "high"
            issue_type = "missing_mandatory_values"
        else:
            severity = "medium" if missing_ratio >= 0.20 else "low"
            issue_type = "missing_optional_values"

        issues.append(make_issue(
            dataset_name, tool, issue_type, severity,
            f"Column `{col}` has {effective_missing} effectively missing values ({missing_ratio:.1%}).",
            columns=[col], row_count=effective_missing,
            evidence={"real_nulls": real_nulls, "disguised_nulls": disguised_nulls,
                      "missing_ratio": round(missing_ratio, 4), "samples": safe_samples(df[col])},
            suggested_fix="Impute, standardize, or drop depending on business criticality."
        ))

    return build_result(dataset_name, tool, issues)

# Flags any column whose completeness (share of non-missing values) falls below a
# configurable threshold; useful for quickly surfacing near-empty columns that
# the per-column null check might bury.
def check_sparse_columns(df, dataset_name, threshold=0.85):
    tool = "check_sparse_columns"
    issues = []

    for col in df.columns:
        total = len(df)
        real_nulls = int(df[col].isna().sum())
        disguised_nulls = int(df[col].astype(str).str.strip().str.lower().isin(DISGUISED_NULLS).sum())
        effective_missing = real_nulls + disguised_nulls
        completeness = 1 - (effective_missing / total if total else 0)

        if completeness < threshold:
            severity = "high" if completeness < 0.60 else "medium"
            issues.append(make_issue(
                dataset_name, tool, "sparse_column", severity,
                f"Column `{col}` completeness is only {completeness:.1%}.",
                columns=[col], row_count=effective_missing,
                evidence={"completeness": round(completeness, 4),
                          "real_nulls": real_nulls, "disguised_nulls": disguised_nulls},
                suggested_fix="Assess whether the column should be imputed, deprecated, or excluded."
            ))

    return build_result(dataset_name, tool, issues, meta={"threshold": threshold})


## 3.3 — Format Consistency Tools

A value can be present but still unusable if its format is inconsistent.  
This cell validates temporal fields, month and year encodings, and low-cardinality categorical columns where casing or spelling variants may hide duplicate categories.

In [ ]:
FORMAT_RULES = {}  # populated by discover_dataset_rules at runtime

# Classifies a single cell value against the expected temporal kind
# (year, month, period, date, or datetime) and returns a label like
# "yyyy-mm-dd", "invalid", or "missing" to build a format distribution.
def classify_format(value, kind):
    if pd.isna(value) or normalize_text(value) is None or is_disguised_null(value):
        return "missing"

    text = str(value).strip()

    if kind == "year":
        return "yyyy" if re.fullmatch(r"(19|20)\d{2}", text) else "invalid"

    if kind == "month":
        return "m_or_mm" if re.fullmatch(r"(0?[1-9]|1[0-2])", text) else "invalid"

    if kind == "period":
        if re.fullmatch(r"\d{4}-\d{2}", text): return "yyyy-mm"
        if re.fullmatch(r"\d{2}/\d{4}", text): return "mm/yyyy"
        if re.fullmatch(r"\d{4}/\d{2}", text): return "yyyy/mm"
        return "invalid"

    if kind == "date":
        if re.fullmatch(r"\d{4}-\d{2}-\d{2}", text): return "yyyy-mm-dd"
        if re.fullmatch(r"\d{2}/\d{2}/\d{4}", text): return "dd/mm/yyyy"
        if pd.to_datetime(pd.Series([text]), errors="coerce", dayfirst=True).notna().iloc[0]:
            return "other_parseable_date"
        return "invalid"

    if kind == "datetime":
        if pd.to_datetime(pd.Series([text]), errors="coerce", dayfirst=True).notna().iloc[0]:
            return "datetime_like" if ("T" in text or ":" in text) else "date_like"
        return "invalid"

    return "unknown"

# Scans columns listed in FORMAT_RULES and raises an issue when invalid values
# exist or when multiple valid format families are mixed in the same column
# (e.g., some dates as "dd/mm/yyyy" and others as "yyyy-mm-dd").
def check_formats(df, dataset_name, format_rules=FORMAT_RULES):
    tool = "check_formats"
    issues = []
    rules = format_rules.get(dataset_name, {})

    for col, kind in rules.items():
        if col not in df.columns:
            continue

        classified = df[col].apply(lambda x: classify_format(x, kind))
        non_missing = classified[classified != "missing"]

        if len(non_missing) == 0:
            continue

        invalid_count = int((non_missing == "invalid").sum())
        valid_share = 1 - (invalid_count / len(non_missing))
        format_mix = non_missing[non_missing != "invalid"].value_counts().to_dict()

        if invalid_count > 0:
            severity = "high" if valid_share < 0.80 else "medium"
            issues.append(make_issue(
                dataset_name, tool, "invalid_format_values", severity,
                f"Column `{col}` contains {invalid_count} invalid {kind} values.",
                columns=[col], row_count=invalid_count,
                evidence={"kind": kind, "valid_share": round(valid_share, 4),
                          "format_distribution": format_mix, "samples": safe_samples(df[col])},
                suggested_fix="Normalize this field to one accepted format before validation or joins."
            ))

        if len(format_mix) > 1:
            issues.append(make_issue(
                dataset_name, tool, "mixed_format_family", "medium",
                f"Column `{col}` mixes multiple valid format families: {list(format_mix.keys())}.",
                columns=[col], row_count=len(non_missing),
                evidence={"kind": kind, "format_distribution": format_mix},
                suggested_fix="Convert all values to a single canonical format."
            ))

    return build_result(dataset_name, tool, issues)

# Looks for casing or spelling variants that map to the same lowercase token
# (e.g., "Roma", "ROMA", "roma") in low-cardinality string columns, which would
# inflate group counts in any downstream aggregation.
def check_categorical_case_variants(df, dataset_name, max_unique=40):
    tool = "check_categorical_case_variants"
    issues = []
    object_cols = df.select_dtypes(include="object").columns.tolist()

    for col in object_cols:
        s = df[col].dropna().astype(str).str.strip()
        s = s[s.ne("")]
        if len(s) == 0 or s.nunique() > max_unique:
            continue

        grouped = defaultdict(set)
        for val in s.unique():
            grouped[val.lower()].add(val)

        suspicious = {k: sorted(list(v)) for k, v in grouped.items() if len(v) > 1}
        if suspicious:
            issues.append(make_issue(
                dataset_name, tool, "categorical_variant_inconsistency", "low",
                f"Column `{col}` contains casing or spelling variants that may represent the same category.",
                columns=[col],
                row_count=sum(len(v) for v in suspicious.values()),
                evidence={"variant_groups": suspicious},
                suggested_fix="Standardize category labels before aggregation or modeling."
            ))

    return build_result(dataset_name, tool, issues)

## 3.4 — Numeric Validity, Outliers, and Duplicates

This block separates structural numeric problems from statistical anomalies.  
First we detect invalid numeric encodings and impossible values; then we use IQR (interquartile range) to flag extreme observations; finally we check exact duplicate rows and optional duplicate business keys.

In [ ]:
NUMERIC_RULES = {}  # populated by discover_dataset_rules at runtime

DUPLICATE_KEY_RULES = {}  # populated by discover_dataset_rules at runtime

# Validates numeric columns by detecting values that cannot be parsed to a number
# even after loose cleaning, forbidden tokens like "€", and values that violate
# a known minimum threshold (e.g., negative counts or costs).
def check_numeric_validity(df, dataset_name, numeric_rules=NUMERIC_RULES):
    tool = "check_numeric_validity"
    issues = []
    rules = numeric_rules.get(dataset_name, {})

    for col, rule in rules.items():
        if col not in df.columns:
            continue

        raw = df[col]
        raw_text = raw.astype(str).str.strip()
        parsed = coerce_numeric_loose(raw)
        present_mask = raw.notna() & raw_text.ne("") & ~raw_text.str.lower().isin(DISGUISED_NULLS)
        invalid_mask = present_mask & parsed.isna()

        if invalid_mask.any():
            issues.append(make_issue(
                dataset_name, tool, "non_numeric_values_in_numeric_field", "high",
                f"Column `{col}` contains non-numeric values in a numeric field.",
                columns=[col], row_count=int(invalid_mask.sum()),
                evidence={"samples": raw[invalid_mask].astype(str).drop_duplicates().head(10).tolist()},
                suggested_fix="Strip symbols/text and coerce to numeric using a canonical parser."
            ))

        for token in rule.get("forbid_tokens", []):
            token_mask = raw.astype(str).str.contains(re.escape(token), na=False)
            if token_mask.any():
                issues.append(make_issue(
                    dataset_name, tool, "forbidden_token_in_numeric_field", "medium",
                    f"Column `{col}` contains forbidden token `{token}`.",
                    columns=[col], row_count=int(token_mask.sum()),
                    evidence={"token": token, "samples": raw[token_mask].astype(str).drop_duplicates().head(10).tolist()},
                    suggested_fix="Store numeric values without currency/unit symbols."
                ))

        min_value = rule.get("min")
        if min_value is not None:
            negative_mask = parsed.notna() & (parsed < min_value)
            if negative_mask.any():
                issues.append(make_issue(
                    dataset_name, tool, "value_below_minimum", "high",
                    f"Column `{col}` contains values below the allowed minimum ({min_value}).",
                    columns=[col], row_count=int(negative_mask.sum()),
                    evidence={"min_allowed": min_value,
                              "samples": raw[negative_mask].astype(str).drop_duplicates().head(10).tolist()},
                    suggested_fix="Review source data or clamp/remove impossible values."
                ))

    return build_result(dataset_name, tool, issues)

# Uses the Interquartile Range (IQR) method to flag statistical outliers in numeric
# columns. The whisker multiplier (default 1.5) controls sensitivity; only columns
# with enough data variety are tested to avoid false positives on near-constant fields.
def check_outliers_iqr(df, dataset_name, numeric_rules=NUMERIC_RULES, whisker=1.5):
    tool = "check_outliers_iqr"
    issues = []
    candidate_cols = [c for c in numeric_rules.get(dataset_name, {}) if c in df.columns]

    for col in candidate_cols:
        vals = coerce_numeric_loose(df[col]).dropna()
        if len(vals) < 8 or vals.nunique() < 4:
            continue

        q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue

        lower, upper = q1 - whisker * iqr, q3 + whisker * iqr
        outlier_mask = coerce_numeric_loose(df[col]).notna() & ~coerce_numeric_loose(df[col]).between(lower, upper)
        outlier_count = int(outlier_mask.sum())

        if outlier_count > 0:
            severity = "medium" if outlier_count / len(df) < 0.05 else "high"
            issues.append(make_issue(
                dataset_name, tool, "iqr_outliers", severity,
                f"Column `{col}` contains {outlier_count} IQR-based outliers.",
                columns=[col], row_count=outlier_count,
                evidence={"q1": float(q1), "q3": float(q3), "iqr": float(iqr),
                          "lower_bound": float(lower), "upper_bound": float(upper),
                          "samples": df.loc[outlier_mask, col].astype(str).drop_duplicates().head(10).tolist()},
                suggested_fix="Review whether these are valid spikes or ingestion errors."
            ))

    return build_result(dataset_name, tool, issues, meta={"whisker": whisker})

# Detects fully identical duplicate rows (exact copies) and, when business-key
# columns are configured, also finds key duplicates that carry divergent data —
# a more dangerous form of duplication that silently inflates aggregations.
def check_duplicates(df, dataset_name, duplicate_key_rules=DUPLICATE_KEY_RULES):
    tool = "check_duplicates"
    issues = []

    exact_dup_mask = df.duplicated(keep=False)
    if exact_dup_mask.any():
        issues.append(make_issue(
            dataset_name, tool, "exact_duplicate_rows", "medium",
            "Exact duplicate rows detected.",
            columns=list(df.columns), row_count=int(exact_dup_mask.sum()),
            evidence={"sample_rows": df[exact_dup_mask].head(5).to_dict(orient="records")},
            suggested_fix="Deduplicate exact copies before aggregation or scoring."
        ))

    key_cols = [c for c in duplicate_key_rules.get(dataset_name, []) if c in df.columns]
    if key_cols:
        key_dup_mask = df.duplicated(subset=key_cols, keep=False)
        if key_dup_mask.any():
            issues.append(make_issue(
                dataset_name, tool, "duplicate_business_keys", "high",
                f"Duplicate business keys detected on {key_cols}.",
                columns=key_cols, row_count=int(key_dup_mask.sum()),
                evidence={"sample_rows": df[key_dup_mask].head(5).to_dict(orient="records")},
                suggested_fix="Investigate whether repeated keys represent conflicts or valid snapshots."
            ))

    return build_result(dataset_name, tool, issues)


## 3.5 — Cross-Column Logic Rules

Some errors only appear when columns are evaluated together.  
This tool enforces dataset-specific business logic such as year/date agreement, month-year alignment, and logical count constraints like `INVESTIGATI <= ENTRATI`.

In [ ]:
# Parses a string that should contain a 4-digit year; returns np.nan on failure.
def parse_year(text):
    if pd.isna(text) or normalize_text(text) is None or is_disguised_null(text):
        return np.nan
    text = str(text).strip()
    return int(text) if re.fullmatch(r"(19|20)\d{2}", text) else np.nan

# Parses a period string (yyyy-mm, mm/yyyy, yyyy/mm) into a (year, month) tuple;
# returns (nan, nan) when the format is unrecognized.
def parse_year_month(text):
    if pd.isna(text) or normalize_text(text) is None or is_disguised_null(text):
        return (np.nan, np.nan)
    text = str(text).strip()
    if re.fullmatch(r"\d{4}-\d{2}", text):
        y, m = text.split("-"); return int(y), int(m)
    if re.fullmatch(r"\d{4}/\d{2}", text):
        y, m = text.split("/"); return int(y), int(m)
    if re.fullmatch(r"\d{2}/\d{4}", text):
        m, y = text.split("/"); return int(y), int(m)
    return (np.nan, np.nan)

# Attempts to parse a date string and extracts only the year component;
# returns np.nan when the string cannot be parsed as a date.
def parse_date_year(text):
    if pd.isna(text) or normalize_text(text) is None or is_disguised_null(text):
        return np.nan
    dt = pd.to_datetime(pd.Series([str(text).strip()]), errors="coerce", dayfirst=True).iloc[0]
    return dt.year if pd.notna(dt) else np.nan

# Enforces dataset-specific multi-column business rules:
#   - tipologia:   INVESTIGATI must not exceed ENTRATI (logical count ceiling)
#   - tipologia/allarmi: year extracted from DATA_PARTENZA must match ANNO_PARTENZA
#   - attivazioni: mese and anno must be consistent with the RATA period string
def check_cross_column(df, dataset_name):
    tool = "check_cross_column"
    issues = []

    if dataset_name == "tipologia":
        if {"ENTRATI", "INVESTIGATI"}.issubset(df.columns):
            entrati = coerce_numeric_loose(df["ENTRATI"])
            investigati = coerce_numeric_loose(df["INVESTIGATI"])
            bad = investigati.notna() & entrati.notna() & (investigati > entrati)
            if bad.any():
                issues.append(make_issue(
                    dataset_name, tool, "logical_count_violation", "critical",
                    "`INVESTIGATI` cannot be greater than `ENTRATI`.",
                    columns=["INVESTIGATI", "ENTRATI"], row_count=int(bad.sum()),
                    evidence={"sample_rows": df.loc[bad, ["INVESTIGATI", "ENTRATI"]].head(10).to_dict(orient="records")},
                    suggested_fix="Review row-level counts and restore logical consistency."
                ))

    if dataset_name in {"tipologia", "allarmi"}:
        if {"DATA_PARTENZA", "ANNO_PARTENZA"}.issubset(df.columns):
            data_year = df["DATA_PARTENZA"].apply(parse_date_year)
            anno = df["ANNO_PARTENZA"].apply(parse_year)
            bad = data_year.notna() & anno.notna() & (data_year != anno)
            if bad.any():
                issues.append(make_issue(
                    dataset_name, tool, "year_date_mismatch", "high",
                    "`ANNO_PARTENZA` does not match the year extracted from `DATA_PARTENZA`.",
                    columns=["DATA_PARTENZA", "ANNO_PARTENZA"], row_count=int(bad.sum()),
                    evidence={"sample_rows": df.loc[bad, ["DATA_PARTENZA", "ANNO_PARTENZA"]].head(10).to_dict(orient="records")},
                    suggested_fix="Derive one field from the other or standardize both from source."
                ))

    if dataset_name == "attivazioni":
        if {"mese", "anno", "RATA"}.issubset(df.columns):
            ym_from_rata = df["RATA"].apply(parse_year_month)
            rata_year  = ym_from_rata.apply(lambda x: x[0])
            rata_month = ym_from_rata.apply(lambda x: x[1])
            anno = df["anno"].apply(parse_year)
            mese = pd.to_numeric(df["mese"], errors="coerce")
            bad = anno.notna() & mese.notna() & rata_year.notna() & rata_month.notna() & (
                (anno != rata_year) | (mese != rata_month)
            )
            if bad.any():
                issues.append(make_issue(
                    dataset_name, tool, "month_year_period_mismatch", "high",
                    "`mese` and `anno` are inconsistent with `RATA`.",
                    columns=["mese", "anno", "RATA"], row_count=int(bad.sum()),
                    evidence={"sample_rows": df.loc[bad, ["mese", "anno", "RATA"]].head(10).to_dict(orient="records")},
                    suggested_fix="Rebuild the period fields from one canonical temporal source."
                ))

    return build_result(dataset_name, tool, issues)

## 3.6 — Tool Registry and Dataset Runner

The last step, wires all deterministic tools into one execution pipeline.  
This produces two outputs: a nested dictionary for agent consumption and a flat issues table for quick human inspection.

In [ ]:
PHASE3_TOOLS = [
    check_schema,
    check_nulls,
    check_sparse_columns,
    check_formats,
    check_categorical_case_variants,
    check_numeric_validity,
    check_outliers_iqr,
    check_duplicates,
    check_cross_column
]

# Keyed by function name so LangGraph agents can call tools by string reference.
agent_tools = {tool.__name__: tool for tool in PHASE3_TOOLS}

# Runs every tool in PHASE3_TOOLS against a single DataFrame and collects their results.
def audit_dataset(df, dataset_name, auto_discover=True):
    if auto_discover and dataset_name not in EXPECTED_SCHEMAS:
        discover_dataset_rules(df, dataset_name)

    return {
        "dataset": dataset_name,
        "results": [tool(df, dataset_name) for tool in PHASE3_TOOLS]
    }

# Explodes all nested issue dicts across all datasets into a single flat DataFrame
# for easy sorting, filtering, and display.
def flatten_issues(audit_output):
    rows = []
    for dataset_name, payload in audit_output.items():
        for result in payload["results"]:
            rows.extend(result["issues"])
    return pd.DataFrame(rows)

# Produces a one-row-per-tool summary table showing issue counts and
# severity breakdowns — useful as a quick health check before agent reasoning.
def flatten_summary(audit_output):
    rows = []
    for dataset_name, payload in audit_output.items():
        for result in payload["results"]:
            rows.append({
                "dataset": dataset_name,
                "tool": result["tool"],
                "issue_count": result["issue_count"],
                "severity_breakdown": json.dumps(result["severity_breakdown"]),
            })
    return pd.DataFrame(rows)

print(f"Phase 3 ready. Tools: {[t.__name__ for t in PHASE3_TOOLS]}")


## Phase 4 — Benchmark sintetico (validazione del layer deterministico)

Iniziamo da un sample clean di ogni dataset, iniettiamo errori controllati di **3 tipi rappresentativi** (`disguised_null`, `iqr_outlier`, `exact_duplicate`) e misuriamo Precision/Recall/F1 a livello di coppia `(dataset, error_type)`. Una sola figura: bar chart delle 3 metriche globali.

Lo scopo è **mostrare che il layer deterministico è validato** prima di delegarne il ragionamento agli agenti LLM. Niente over-engineering: 3 tipi di errore coprono i casi rappresentativi (categorica, numerica, strutturale).


In [ ]:
import random
import matplotlib.pyplot as plt
from pathlib import Path

random.seed(42); np.random.seed(42)

BENCHMARK_DIR = Path("data/benchmark"); BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
(BENCHMARK_DIR / "charts").mkdir(exist_ok=True)
Path("images").mkdir(exist_ok=True)

# 3 error types — we map each to the set of deterministic issue_types that should catch it.
ERROR_DETECTORS = {
    "disguised_null":  {"missing_mandatory_values", "missing_optional_values", "sparse_column"},
    "iqr_outlier":     {"iqr_outliers"},
    "exact_duplicate": {"exact_duplicate_rows"},
}


def inject_and_evaluate(df, name, n_each=3):
    """Inject n_each errors per type, run audit, return (gt, detected) pairs."""
    df = df.sample(min(500, len(df)), random_state=42).reset_index(drop=True).copy()
    gt = set()

    str_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

    for _ in range(n_each):
        # disguised_null — target only string columns to avoid corrupting numeric dtypes
        if str_cols:
            df.at[random.randint(0, len(df)-1), random.choice(str_cols)] = "N.D."
            gt.add((name, "disguised_null"))

        # iqr_outlier — multiply max by 100 to push it well outside the IQR fence
        if num_cols:
            col = random.choice(num_cols)
            big = float(pd.to_numeric(df[col], errors="coerce").max()) * 100 + 1_000_000
            df.at[random.randint(0, len(df)-1), col] = big
            gt.add((name, "iqr_outlier"))

        # exact_duplicate — clone a random row
        df = pd.concat([df, df.iloc[[random.randint(0, len(df)-1)]]], ignore_index=True)
        gt.add((name, "exact_duplicate"))

    discover_dataset_rules(df, name)
    audit = audit_dataset(df, name, auto_discover=False)

    detected = set()
    for r in audit["results"]:
        for issue in r["issues"]:
            for et, expected in ERROR_DETECTORS.items():
                if issue["issue_type"] in expected:
                    detected.add((name, et))
    return gt, detected


# Aggregate across the 4 test fixtures
all_gt, all_det = set(), set()
for name, df in datasets.items():
    gt, det = inject_and_evaluate(df, name)
    all_gt |= gt
    all_det |= det

tp, fp, fn = all_gt & all_det, all_det - all_gt, all_gt - all_det
P = len(tp) / (len(tp) + len(fp)) if (tp or fp) else 0.0
R = len(tp) / (len(tp) + len(fn)) if (tp or fn) else 0.0
F1 = 2 * P * R / (P + R) if (P + R) else 0.0

results = {"TP": len(tp), "FP": len(fp), "FN": len(fn),
           "Precision": round(P, 3), "Recall": round(R, 3), "F1": round(F1, 3)}
(BENCHMARK_DIR / "evaluation_results.json").write_text(json.dumps(results, indent=2))
print(f"Benchmark on 4 datasets × 3 error types: {results}")

# Single P/R/F1 bar chart, saved to both data/benchmark and images/ for the README
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Precision", "Recall", "F1"], [P, R, F1], color=["#4c72b0", "#55a868", "#c44e52"])
ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
ax.set_title("Deterministic layer — synthetic benchmark")
for i, v in enumerate([P, R, F1]):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center")
plt.tight_layout()
plt.savefig(BENCHMARK_DIR / "charts" / "benchmark_metrics.png", dpi=120)
plt.savefig("images/detection_heatmap.png", dpi=120)
plt.show()


## Phase 5 — Multi-Agent Pipeline (LangGraph)

L'architettura è **Determinism-first con LLM chirurgico**: il layer deterministico (Phase 3) consolida tutti gli issue rilevabili da regole; un singolo agente LLM (`RemediationPlanner`) decide *quale* azione di fix applicare per ogni gruppo di issue; un esecutore deterministico applica i fix con tool atomici. Il LLM interviene solo dove la decisione richiede ragionamento contestuale.

### State graph
```
ingest → audit_deterministic → remediation_planner (LLM) → remediation_executor → reliability_score
```

Lo stato condiviso (`AgentState`) accumula: dataset name, raw df, issues, remediation plan, fixed df, correction log, reliability score.


### 5.1 — Atomic remediation tools

10 funzioni pure di tipo `(df, col, **kwargs) → (df_fixed, log_entry)`. Sono il 'Phase 6' del piano: building blocks deterministici che il `RemediationPlanner` orchestra.


In [ ]:
# ─── Atomic remediation tools — pure functions, returning (df, log_entry) ──
# Each fix is short and self-contained. The log_entry uses a uniform shape
# {action, column(s), rows_affected, applied, ...} so the report renders easily.

def _log(action, applied=True, **extra):
    out = {"action": action, "applied": applied}
    out.update(extra)
    return out

def _missing(action, col):
    return None, _log(action, applied=False, column=col, reason="column missing")


def fix_strip_currency(df, col, **_):
    if col not in df.columns: return df, _log("strip_currency", False, column=col, reason="column missing")
    before = df[col].astype(str).str.contains(r"[€$£]", na=False).sum()
    df = df.copy()
    df[col] = df[col].astype(str).str.replace(r"[€$£]", "", regex=True).str.strip()
    df[col] = pd.to_numeric(df[col].str.replace(",", ".", regex=False), errors="ignore")
    return df, _log("strip_currency", column=col, rows_affected=int(before))


def fix_cast_numeric(df, col, **_):
    if col not in df.columns: return df, _log("cast_numeric", False, column=col, reason="column missing")
    df = df.copy()
    new = coerce_numeric_loose(df[col])
    n = int((df[col].astype(str) != new.astype(str)).sum())
    df[col] = new
    return df, _log("cast_numeric", column=col, rows_affected=n)


def fix_impute_median(df, col, **_):
    if col not in df.columns: return df, _log("impute_median", False, column=col, reason="column missing")
    df = df.copy()
    s = coerce_numeric_loose(df[col])
    median = s.median()
    if pd.isna(median): return df, _log("impute_median", False, column=col, reason="no numeric data")
    mask = s.isna() | df[col].astype(str).str.strip().str.lower().isin(DISGUISED_NULLS)
    df.loc[mask, col] = median
    return df, _log("impute_median", column=col, rows_affected=int(mask.sum()), value=float(median))


def fix_impute_mode(df, col, **_):
    if col not in df.columns: return df, _log("impute_mode", False, column=col, reason="column missing")
    df = df.copy()
    s = df[col].astype(str).str.strip()
    clean = s[~s.str.lower().isin(DISGUISED_NULLS) & df[col].notna()]
    if clean.empty: return df, _log("impute_mode", False, column=col, reason="all values missing")
    mode = clean.mode().iloc[0]
    mask = df[col].isna() | s.str.lower().isin(DISGUISED_NULLS)
    df.loc[mask, col] = mode
    return df, _log("impute_mode", column=col, rows_affected=int(mask.sum()), value=str(mode))


def fix_clip_iqr(df, col, **_):
    if col not in df.columns: return df, _log("clip_iqr", False, column=col, reason="column missing")
    df = df.copy()
    s = coerce_numeric_loose(df[col])
    if s.dropna().empty: return df, _log("clip_iqr", False, column=col, reason="no numeric data")
    q1, q3 = s.quantile([0.25, 0.75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    mask = s.notna() & ~s.between(lo, hi)
    df.loc[mask, col] = s[mask].clip(lo, hi)
    return df, _log("clip_iqr", column=col, rows_affected=int(mask.sum()),
                    bounds={"lower": float(lo), "upper": float(hi)})


def fix_drop_duplicates(df, **_):
    df = df.copy()
    n_before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    return df, _log("drop_duplicates", rows_affected=n_before - len(df))


def fix_normalize_dates(df, col, target="%Y-%m-%d", **_):
    if col not in df.columns: return df, _log("normalize_dates", False, column=col, reason="column missing")
    df = df.copy()
    parsed = pd.to_datetime(df[col].astype(str).str.strip(), errors="coerce", dayfirst=True)
    df[col] = parsed.dt.strftime(target).where(parsed.notna(), df[col])
    return df, _log("normalize_dates", column=col, rows_affected=int(parsed.notna().sum()))


def fix_drop_unexpected_columns(df, columns=None, **_):
    cols = [c for c in (columns or []) if c in df.columns]
    df = df.copy().drop(columns=cols)
    return df, _log("drop_unexpected_columns", columns=cols, rows_affected=len(df))


def fix_normalize_categorical(df, col, **_):
    if col not in df.columns: return df, _log("normalize_categorical", False, column=col, reason="column missing")
    df = df.copy()
    df[col] = df[col].astype(str).str.strip().str.lower().str.title()
    return df, _log("normalize_categorical", column=col, rows_affected=len(df))


def fix_ignore(df, col=None, **_):
    return df, _log("ignore", column=col, reason="deferred or low-priority")


REMEDIATION_TOOLS = {
    "strip_currency":          fix_strip_currency,
    "cast_numeric":            fix_cast_numeric,
    "impute_median":           fix_impute_median,
    "impute_mode":             fix_impute_mode,
    "clip_iqr":                fix_clip_iqr,
    "drop_duplicates":         fix_drop_duplicates,
    "normalize_dates":         fix_normalize_dates,
    "drop_unexpected_columns": fix_drop_unexpected_columns,
    "normalize_categorical":   fix_normalize_categorical,
    "ignore":                  fix_ignore,
}
print(f"Loaded {len(REMEDIATION_TOOLS)} remediation tools.")


### 5.2 — AgentState + ingest/discover/audit

Lo stato è un `TypedDict` con accumulatori (`Annotated[..., operator.add]` per liste, `_merge_dicts` per i sub-scores). I primi tre nodi sono **deterministici**: caricano il df, scoprono regole con euristiche, eseguono i 9 tool di Phase 3.


In [ ]:
# ─── AgentState (LangGraph TypedDict) ─────────────────────────────────────
# Aggregators for parallel-friendly accumulation across agents:
def _merge_dicts(a, b): return {**a, **b}

class AgentState(TypedDict, total=False):
    dataset_name: str
    raw_df: Any
    issues: List[Dict[str, Any]]
    severity_breakdown: Dict[str, int]
    plan: Annotated[List[Dict[str, Any]], operator.add]   # accumulates from each analysis agent
    sub_scores: Annotated[Dict[str, float], _merge_dicts] # 5 dimensions: validity/completeness/consistency/uniqueness/accuracy
    fixed_df: Any
    correction_log: List[Dict[str, Any]]
    reliability_score: float
    audit_trail: Annotated[List[str], operator.add]       # narrative trace from each agent
    errors: Annotated[List[str], operator.add]


def node_ingest(state: AgentState) -> Dict[str, Any]:
    """Load the dataframe (from datasets registry or already-passed df)."""
    name = state["dataset_name"]
    df = state.get("raw_df")
    if df is None:
        df = datasets[name]
    return {"raw_df": df.copy(), "audit_trail": [f"ingest: loaded `{name}` ({df.shape[0]:,}x{df.shape[1]})"], "errors": []}


def node_discover(state: AgentState) -> Dict[str, Any]:
    """Discover schema/mandatory/format/numeric rules from the dataframe sample.
    Heuristic-only — no LLM call here, to keep the discovery deterministic and free."""
    name = state["dataset_name"]
    df = state["raw_df"]
    rules = discover_dataset_rules(df, name)
    msg = (f"discover: schema_cols={len(rules['schema']['required_columns'])} "
           f"mandatory={len(rules['mandatory'])} dates={len(rules['format_rules'])} "
           f"numeric={len(rules['numeric_rules'])}")
    return {"audit_trail": [msg]}


def node_audit(state: AgentState) -> Dict[str, Any]:
    """Run all 9 deterministic tools and collect issues."""
    name = state["dataset_name"]
    df = state["raw_df"]
    audit = audit_dataset(df, name, auto_discover=False)  # rules already populated
    flat = []
    for r in audit["results"]:
        flat.extend(r["issues"])
    sev = {"critical": 0, "high": 0, "medium": 0, "low": 0}
    for issue in flat:
        sev[issue["severity"]] = sev.get(issue["severity"], 0) + 1
    msg = f"audit: {len(flat)} issues — {sev}"
    return {"issues": flat, "severity_breakdown": sev, "audit_trail": [msg]}


### 5.3 — 4 LLM analysis agents (Schema, Completeness, Consistency, Anomaly)

Ogni agente riceve la *propria fetta* di issue + un **fact pack di colonne** (dtype, samples, n_unique, n_nulls) per le colonne coinvolte. Questo permette al LLM di ragionare concretamente sui dati reali invece di indovinare. Output: piano JSON con `why` come *osservazione concreta* (es. *"14 N.D. tokens, 2.8% — lieve, imputable with mode"*).

Token budget per agente: 600-1200 (più ricco grazie al column context). Totale ~3-5k per dataset. Fallback rule-based deterministico se l'LLM fallisce.


In [ ]:
# ─── 4 LLM analysis agents — focused, contextual, structured ──────────────
# Each agent receives:
#   • its slice of issues (filtered by issue_type)
#   • column-level context (samples + stats) for the columns involved
# and returns a plan slice + a 0-1 sub-score for its dimension(s).

SCHEMA_ISSUE_TYPES       = {"missing_required_columns", "duplicate_column_names",
                            "semantic_type_mismatch", "naming_convention_violation"}
COMPLETENESS_ISSUE_TYPES = {"missing_mandatory_values", "missing_optional_values", "sparse_column"}
CONSISTENCY_ISSUE_TYPES  = {"invalid_format_values", "mixed_format_family",
                            "categorical_variant_inconsistency",
                            "exact_duplicate_rows", "duplicate_business_keys",
                            "year_date_mismatch", "month_year_period_mismatch",
                            "logical_count_violation"}
ANOMALY_ISSUE_TYPES      = {"iqr_outliers", "non_numeric_values_in_numeric_field",
                            "forbidden_token_in_numeric_field", "value_below_minimum"}

SEVERITY_PENALTY = {"critical": 0.40, "high": 0.20, "medium": 0.08, "low": 0.02}


def _compute_subscore(issues):
    score = 1.0
    for iss in issues:
        score -= SEVERITY_PENALTY.get(iss["severity"], 0.05)
    return max(0.0, round(score, 3))


def _column_context(df, columns, max_samples=5):
    """Compact dict of {col: {dtype, n_unique, n_nulls, samples}} — only for columns referenced by the issues.
    This is the 'fact pack' the LLM uses to reason concretely instead of hallucinating."""
    ctx = {}
    for col in set(columns):
        if col not in df.columns:
            continue
        s = df[col]
        samples = s.dropna().astype(str).str.strip()
        samples = samples[samples.ne("")].drop_duplicates().head(max_samples).tolist()
        ctx[col] = {
            "dtype": str(s.dtype),
            "n_unique": int(s.nunique(dropna=True)),
            "n_nulls": int(s.isna().sum()),
            "samples": samples,
        }
    return ctx


def _safe_parse_json_list(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.startswith("json"): text = text[4:].strip()
    s, e = text.find("["), text.rfind("]")
    if s >= 0 and e > s:
        try: return json.loads(text[s:e+1])
        except Exception: pass
    return None


_FALLBACK = {
    "duplicate_column_names": "drop_unexpected_columns",
    "semantic_type_mismatch": "cast_numeric",
    "missing_mandatory_values": "impute_mode",
    "invalid_format_values": "normalize_dates",
    "mixed_format_family": "normalize_dates",
    "categorical_variant_inconsistency": "normalize_categorical",
    "exact_duplicate_rows": "drop_duplicates",
    "duplicate_business_keys": "drop_duplicates",
    "iqr_outliers": "clip_iqr",
    "value_below_minimum": "clip_iqr",
    "non_numeric_values_in_numeric_field": "cast_numeric",
    "forbidden_token_in_numeric_field": "strip_currency",
}


def _make_agent(name, issue_types, allowed, dims):
    """Build an analysis-agent node with a focused, contextual prompt."""
    actions_str = ", ".join(allowed)
    sys_prompt = (
        f"You are the {name} agent in a data quality multi-agent system.\n"
        f"Scope: issues of type {sorted(issue_types)}.\n"
        f"Allowed actions: {actions_str}.\n\n"
        "For each issue you receive, decide the best action. Use the column context "
        "(dtype, samples, null counts) to reason concretely, not abstractly.\n\n"
        "Output STRICT JSON: an array of objects, one per input issue in the same order:\n"
        '  [{"i": <issue_index>, "action": "<allowed_name>", "why": "<concrete observation, max 25 words>"}]\n'
        "No markdown fencing, no prose outside the JSON."
    )

    def node(state: AgentState) -> Dict[str, Any]:
        issues = state.get("issues", [])
        relevant = [(i, iss) for i, iss in enumerate(issues) if iss["issue_type"] in issue_types]

        sub_score = _compute_subscore([iss for _, iss in relevant])
        sub_scores = {k: sub_score for k in dims}

        if not relevant:
            return {"plan": [], "sub_scores": sub_scores,
                    "audit_trail": [f"{name}: no issues in scope (score={sub_score})"]}

        df = state["raw_df"]
        cols_involved = [c for _, iss in relevant for c in iss.get("columns", [])]
        col_ctx = _column_context(df, cols_involved)

        compact = [{"i": i, "type": iss["issue_type"], "sev": iss["severity"],
                    "cols": iss.get("columns", []),
                    "msg": (iss.get("message") or "")[:120]}
                   for i, iss in relevant]

        user_msg = (
            f"Dataset: {state['dataset_name']}\n"
            f"Issues to plan ({len(compact)}):\n{json.dumps(compact, ensure_ascii=False)}\n\n"
            f"Column context (referenced by these issues):\n{json.dumps(col_ctx, ensure_ascii=False, default=str)}"
        )

        try:
            resp = llm.invoke([SystemMessage(content=sys_prompt), HumanMessage(content=user_msg)])
            parsed = _safe_parse_json_list(resp.content)
        except Exception as e:
            parsed = None
            err = f"{name}_llm_failed: {type(e).__name__}: {e}"
        else:
            err = None

        if parsed is None or len(parsed) != len(relevant):
            plan = [{"issue_index": i, "action": _FALLBACK.get(iss["issue_type"], "ignore"),
                     "rationale": f"deterministic fallback for {iss['issue_type']}", "agent": name}
                    for i, iss in relevant]
            tag = " (fallback)"
        else:
            plan = []
            for entry, (i, _) in zip(parsed, relevant):
                action = entry.get("action") if entry.get("action") in allowed else "ignore"
                plan.append({"issue_index": i, "action": action,
                             "rationale": (entry.get("why") or "")[:200],
                             "agent": name})
            tag = ""

        out = {"plan": plan, "sub_scores": sub_scores,
               "audit_trail": [f"{name}: {len(plan)} actions planned, score={sub_score}{tag}"]}
        if err: out["errors"] = [err]
        return out

    return node


# Allowed actions per agent — restricted enums keep LLM output sharp and predictable
SCHEMA_ACTIONS       = ["drop_unexpected_columns", "cast_numeric", "normalize_dates", "ignore"]
COMPLETENESS_ACTIONS = ["impute_median", "impute_mode", "ignore"]
CONSISTENCY_ACTIONS  = ["normalize_dates", "drop_duplicates", "normalize_categorical", "ignore"]
ANOMALY_ACTIONS      = ["clip_iqr", "cast_numeric", "strip_currency", "ignore"]

node_schema_agent       = _make_agent("Schema",       SCHEMA_ISSUE_TYPES,       SCHEMA_ACTIONS,       ["validity"])
node_completeness_agent = _make_agent("Completeness", COMPLETENESS_ISSUE_TYPES, COMPLETENESS_ACTIONS, ["completeness"])
node_consistency_agent  = _make_agent("Consistency",  CONSISTENCY_ISSUE_TYPES,  CONSISTENCY_ACTIONS,  ["consistency", "uniqueness"])
node_anomaly_agent      = _make_agent("Anomaly",      ANOMALY_ISSUE_TYPES,      ANOMALY_ACTIONS,      ["accuracy"])


### 5.4 — RemediationAgent + 5-dimension Supervisor

Il `RemediationAgent` applica il piano consolidato (concat di tutte le slice degli analysis agent) usando i tool atomici. Il `SupervisorAgent` è **deterministico**: aggrega i 5 sub-score con i pesi standard ISO-8000 (completeness 30%, consistency 25%, validity 20%, uniqueness 15%, accuracy 10%) e produce un reliability score 0-100. Niente LLM call qui — la matematica è esatta e riproducibile.


In [ ]:
def node_remediation(state: AgentState) -> Dict[str, Any]:
    """Apply the merged plan to the dataframe with atomic deterministic tools."""
    df = state["raw_df"].copy()
    issues = state["issues"]
    plan = state.get("plan", [])
    log = []

    for entry in plan:
        idx = entry.get("issue_index", -1)
        if not (0 <= idx < len(issues)):
            continue
        action = entry["action"]
        tool_fn = REMEDIATION_TOOLS.get(action, fix_ignore)
        cols = issues[idx].get("columns", [])
        col = cols[0] if cols else None
        try:
            df, log_entry = tool_fn(df, col=col, columns=cols)
            log_entry.update({
                "issue_index": idx,
                "issue_type": issues[idx]["issue_type"],
                "rationale": entry.get("rationale", ""),
                "agent": entry.get("agent", "")
            })
            log.append(log_entry)
        except Exception as e:
            log.append({"action": action, "issue_index": idx, "applied": False,
                        "error": f"{type(e).__name__}: {e}",
                        "agent": entry.get("agent", "")})

    n_applied = sum(1 for e in log if e.get("applied"))
    return {"fixed_df": df, "correction_log": log,
            "audit_trail": [f"remediation: applied {n_applied}/{len(log)} fixes"]}


# ─── 5-dimension reliability scoring (ISO-8000-style) ──────────────────────
RELIABILITY_WEIGHTS = {
    "completeness": 0.30,
    "consistency":  0.25,
    "validity":     0.20,
    "uniqueness":   0.15,
    "accuracy":     0.10,
}
assert abs(sum(RELIABILITY_WEIGHTS.values()) - 1.0) < 1e-9


def node_supervisor(state: AgentState) -> Dict[str, Any]:
    """Deterministic supervisor — aggregates 5 sub-scores into a single 0-100 reliability score.
    No LLM call here: the math is exact and reproducible."""
    sub = state.get("sub_scores", {})
    # Default any missing dimension to 1.0 (i.e. no detected issue in that dim)
    final = sum(RELIABILITY_WEIGHTS[k] * sub.get(k, 1.0) for k in RELIABILITY_WEIGHTS) * 100
    final = round(final, 1)
    msg = f"supervisor: reliability={final}/100 sub_scores={sub}"
    return {"reliability_score": final,
            "sub_scores": {k: round(sub.get(k, 1.0)*100, 1) for k in RELIABILITY_WEIGHTS},
            "audit_trail": [msg]}


### 5.5 — StateGraph (linear, 9 nodi, 1 iterazione)

Grafo lineare: `ingest → discover → audit → schema → completeness → consistency → anomaly → remediation → supervisor → END`. Singola iterazione, nessun loop — il feedback del mid-check chiedeva LLM importanti ma non totalizzanti, e il loop di rerun moltiplicherebbe il costo token senza migliorare qualitativamente l'output.


In [ ]:
# ─── Compile the StateGraph (6 nodes, linear, single iteration) ───────────
graph_builder = StateGraph(AgentState)
graph_builder.add_node("ingest",        node_ingest)
graph_builder.add_node("discover",      node_discover)
graph_builder.add_node("audit",         node_audit)
graph_builder.add_node("schema",        node_schema_agent)
graph_builder.add_node("completeness",  node_completeness_agent)
graph_builder.add_node("consistency",   node_consistency_agent)
graph_builder.add_node("anomaly",       node_anomaly_agent)
graph_builder.add_node("remediation",   node_remediation)
graph_builder.add_node("supervisor",    node_supervisor)

graph_builder.add_edge(START, "ingest")
graph_builder.add_edge("ingest", "discover")
graph_builder.add_edge("discover", "audit")
graph_builder.add_edge("audit", "schema")
graph_builder.add_edge("schema", "completeness")
graph_builder.add_edge("completeness", "consistency")
graph_builder.add_edge("consistency", "anomaly")
graph_builder.add_edge("anomaly", "remediation")
graph_builder.add_edge("remediation", "supervisor")
graph_builder.add_edge("supervisor", END)

quality_graph = graph_builder.compile()
print(f"StateGraph compiled: 9 nodes, 4 LLM agents, single iteration.")
print(f"Reliability dimensions: {list(RELIABILITY_WEIGHTS.keys())}")


## Phase 6 — Quality Report (HTML + plotly radar)

Report HTML auto-contenuto con: reliability score gauge, **5 sub-score** (validity/completeness/consistency/uniqueness/accuracy), radar chart plotly interattivo, severity breakdown, audit trail multi-agent, tabella issue + tabella correction log con tracciamento dell'agente che ha proposto ciascun fix. PDF via Print → Save as PDF dal browser.


### 6.1 — Jinja2 template + render function


In [ ]:
from jinja2 import Template

REPORT_TEMPLATE = Template("""
<!DOCTYPE html>
<html lang="it"><head><meta charset="UTF-8">
<title>Data Quality Report — {{ dataset_name }}</title>
<style>
body{font-family:-apple-system,system-ui,sans-serif;max-width:1100px;margin:24px auto;padding:16px;color:#222}
h1{color:#0a6e2c;border-bottom:3px solid #0a6e2c;padding-bottom:8px}
h2{color:#444;margin-top:32px;border-left:4px solid #0a6e2c;padding-left:10px}
.score{display:flex;gap:24px;align-items:center;background:#f4faf6;border:1px solid #cde9d6;padding:18px;border-radius:8px;margin:16px 0}
.gauge{font-size:56px;font-weight:700;color:{{ score_color }}}
.subs{display:grid;grid-template-columns:repeat(5,1fr);gap:10px;margin:14px 0}
.sub{background:#fafafa;border:1px solid #e0e0e0;padding:10px;border-radius:6px;text-align:center}
.sub .l{font-size:11px;color:#666;text-transform:uppercase}
.sub .v{font-size:22px;font-weight:600;color:#0a6e2c}
.sub .w{font-size:10px;color:#999}
table{border-collapse:collapse;width:100%;margin:10px 0;font-size:13px}
th,td{border:1px solid #ddd;padding:6px 10px;text-align:left;vertical-align:top}
th{background:#f0f0f0}
.sev-critical{background:#ffd6d6}.sev-high{background:#ffe9d6}.sev-medium{background:#fff5d6}.sev-low{background:#f0f0f0}
.tag{background:#0a6e2c;color:#fff;padding:2px 8px;border-radius:4px;font-size:11px;font-family:monospace}
.summary{background:#fffdf3;border-left:4px solid #d4ad28;padding:12px 16px;margin:16px 0}
.trail{background:#f4f4f4;border-left:4px solid #888;padding:8px 12px;font-family:monospace;font-size:12px;white-space:pre-wrap}
.muted{color:#888;font-size:12px}
</style>
<script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
</head><body>

<h1>Data Quality Report — {{ dataset_name }}</h1>
<p class="muted">{{ generated_at }} · {{ provider }} · multi-agent pipeline</p>

<div class="score">
  <div class="gauge">{{ reliability_score }}/100</div>
  <div>
    <strong>Reliability score</strong> (ISO-8000 weighted)<br>
    <span class="muted">{{ issues|length }} issues detected · {{ applied_count }} corrections applied</span>
  </div>
</div>

<div class="subs">
{% for cat, val in sub_scores.items() %}
  <div class="sub"><div class="l">{{ cat }}</div><div class="v">{{ val }}</div><div class="w">w={{ weights[cat] }}</div></div>
{% endfor %}
</div>

<div id="radar" style="max-width:480px;margin:16px auto"></div>
<script>
Plotly.newPlot("radar", [{type:"scatterpolar",r:{{ radar_values|tojson }},theta:{{ radar_labels|tojson }},fill:"toself",marker:{color:"#0a6e2c"}}],
{polar:{radialaxis:{visible:true,range:[0,100]}},showlegend:false,margin:{t:20,b:20,l:20,r:20},height:340},{displayModeBar:false});
</script>

<h2>Executive Summary</h2>
<div class="summary">{{ exec_summary | safe }}</div>

<h2>Audit trail</h2>
<div class="trail">{% for line in audit_trail %}{{ line }}\n{% endfor %}</div>

<h2>Top issues</h2>
<table><tr><th>#</th><th>Tool</th><th>Type</th><th>Sev</th><th>Cols</th><th>Rows</th><th>Message</th></tr>
{% for issue in top_issues %}<tr class="sev-{{ issue.severity }}">
<td>{{ loop.index }}</td><td><span class="tag">{{ issue.tool }}</span></td>
<td>{{ issue.issue_type }}</td><td>{{ issue.severity }}</td>
<td>{{ issue.columns | join(", ") }}</td><td>{{ issue.row_count or "-" }}</td>
<td>{{ issue.message }}</td></tr>{% endfor %}</table>

<h2>Correction log ({{ applied_count }} applied)</h2>
<table><tr><th>#</th><th>Agent</th><th>Action</th><th>Issue</th><th>Col(s)</th><th>Rows</th><th>Why</th></tr>
{% for log in correction_log %}<tr>
<td>{{ loop.index }}</td><td>{{ log.agent or "-" }}</td>
<td><span class="tag">{{ log.action }}</span></td><td>{{ log.issue_type or "-" }}</td>
<td>{{ log.columns or log.column or "-" }}</td><td>{{ log.rows_affected or "-" }}</td>
<td>{{ log.rationale or log.reason or "" }}</td></tr>{% endfor %}</table>

<h2>Dataset overview</h2>
<p>Shape: <strong>{{ raw_shape }}</strong> → <strong>{{ fixed_shape }}</strong> after remediation.</p>
{{ sample_html | safe }}

</body></html>
""")


def render_quality_report(state, provider_name="groq llama-3.3-70b"):
    from datetime import datetime
    issues = state.get("issues", [])
    log = state.get("correction_log", [])
    applied = sum(1 for e in log if e.get("applied"))
    score = state.get("reliability_score", 0)
    score_color = "#0a6e2c" if score >= 70 else ("#d4ad28" if score >= 40 else "#c0392b")

    sev_order = {"critical": 0, "high": 1, "medium": 2, "low": 3}
    top_issues = sorted(issues, key=lambda x: (sev_order.get(x["severity"], 4), -(x.get("row_count") or 0)))[:25]

    sub_scores = state.get("sub_scores", {k: 100.0 for k in RELIABILITY_WEIGHTS})
    radar_labels = list(RELIABILITY_WEIGHTS.keys())
    radar_values = [sub_scores.get(k, 100.0) for k in radar_labels]

    raw_df, fixed_df = state["raw_df"], state.get("fixed_df", state["raw_df"])

    return REPORT_TEMPLATE.render(
        dataset_name=state["dataset_name"],
        reliability_score=score, score_color=score_color,
        sub_scores=sub_scores,
        weights={k: f"{int(v*100)}%" for k, v in RELIABILITY_WEIGHTS.items()},
        radar_labels=radar_labels, radar_values=radar_values,
        issues=issues, top_issues=top_issues,
        correction_log=log, applied_count=applied,
        audit_trail=state.get("audit_trail", []),
        raw_shape=f"{raw_df.shape[0]:,} × {raw_df.shape[1]}",
        fixed_shape=f"{fixed_df.shape[0]:,} × {fixed_df.shape[1]}",
        sample_html=fixed_df.head(5).to_html(border=0, index=False),
        provider=provider_name,
        generated_at=datetime.now().strftime("%Y-%m-%d %H:%M"),
        exec_summary=state.get("exec_summary") or _default_exec_summary(state),
    )


def _default_exec_summary(state):
    rs = state.get("reliability_score", 0)
    sev = state.get("severity_breakdown", {})
    n_applied = sum(1 for e in state.get("correction_log", []) if e.get("applied"))
    verdict = "alto" if rs >= 70 else ("medio" if rs >= 40 else "basso")
    return (f"Il dataset <strong>{state['dataset_name']}</strong> ha reliability "
            f"<strong>{rs}/100</strong> (livello {verdict}). Issue: "
            f"{sev.get('critical',0)} critical / {sev.get('high',0)} high / "
            f"{sev.get('medium',0)} medium / {sev.get('low',0)} low. "
            f"Pipeline ha applicato {n_applied} correzioni.")


### 6.2 — Pipeline entry point + optional LLM narrative

`run_quality_pipeline(df, name)` è il punto d'ingresso unico. Riceve un DataFrame qualsiasi, gira la pipeline, opzionalmente aggiunge un summary LLM-narrato, ritorna lo stato finale + HTML report renderizzato. Lo Streamlit chiama esattamente questa funzione.

**Niente auto-run sui 4 dataset.** I CSV in `data/` sono solo *test fixtures*: vengono usati dal benchmark di Phase 4 e da uno smoke-test in Phase 8. Nessun report pre-bakeato viene committato.


In [ ]:
# Optional LLM-narrative for the executive summary (single short call per run)
NARRATIVE_PROMPT = """You are a data quality analyst. In ONE concise paragraph (40-80 words) in Italian,
summarize the data quality of dataset '{name}': mention its reliability score ({score}/100),
the top severity issues, and the most impactful remediation actions taken. Use professional, factual tone.
Output plain text only (no markdown, no headers)."""


def add_llm_narrative(state):
    """Optionally call LLM to produce a narrative executive summary.
    Falls back gracefully if the LLM is unavailable."""
    try:
        prompt = NARRATIVE_PROMPT.format(name=state["dataset_name"], score=state["reliability_score"])
        sev = state.get("severity_breakdown", {})
        sample_issues = "\n".join([f"- {i['issue_type']} ({i['severity']}) on {i.get('columns')}"
                                     for i in state.get("issues", [])[:5]])
        sample_actions = "\n".join([f"- {l.get('action')} on {l.get('column') or l.get('columns')} ({l.get('agent')})"
                                      for l in state.get("correction_log", []) if l.get("applied")][:5])
        ctx = f"Severity breakdown: {sev}\nTop issues:\n{sample_issues}\nApplied actions:\n{sample_actions}"
        resp = llm.invoke([SystemMessage(content=prompt), HumanMessage(content=ctx)])
        state["exec_summary"] = resp.content.strip()
    except Exception:
        state["exec_summary"] = None
    return state


def run_quality_pipeline(df, name="user_dataset", with_narrative=True):
    """Run the full pipeline on a single dataframe and return both the final state and the rendered HTML.
    This is the canonical entry point — used by the notebook smoke test, the Streamlit app,
    and any external caller."""
    # Ensure datasets registry has this entry (so node_ingest can pull it back if df not in state)
    datasets[name] = df
    final_state = quality_graph.invoke({"dataset_name": name, "raw_df": df.copy()})
    if with_narrative:
        final_state = add_llm_narrative(final_state)
    html = render_quality_report(final_state)
    return final_state, html


print("Phase 6 ready. Use `run_quality_pipeline(df, name='your_dataset')` to invoke the pipeline.")


## Phase 7 — Streamlit Demo

App Streamlit auto-contenuta materializzata via `%%writefile app.py`. Due modalità:

1. **Run pipeline su CSV** — l'utente carica un CSV qualsiasi e la pipeline gira live (boot del notebook cached, ~2-3s la prima volta, poi pochi secondi).
2. **Benchmark** — dashboard del Phase 4 (P/R/F1 + heatmap + flowchart).

**Per lanciarla** (dopo aver eseguito Run All del notebook):
```bash
streamlit run agents/app.py
```


In [ ]:
%%writefile app.py
"""Agents for Data Quality — Streamlit demo (LUISS ML 2025/26 · Reply Group 17).

Two modes via the sidebar:
  1. Run pipeline on a CSV — live multi-agent execution with per-node timeline
  2. Benchmark — one-glance view of the deterministic-layer validation
"""
from pathlib import Path
import json, os, sys, time
import pandas as pd
import streamlit as st
import plotly.graph_objects as go

APP_DIR = Path(__file__).parent.resolve()
PROJECT_ROOT = APP_DIR.parent
BENCHMARK_DIR = APP_DIR / "data" / "benchmark"
IMAGES_DIR = APP_DIR / "images"
DATA_DIR = APP_DIR / "data"

st.set_page_config(page_title="Agents for Data Quality", page_icon="🛠️",
                   layout="wide", initial_sidebar_state="expanded")

# ── Lightweight CSS polish ────────────────────────────────────────────────
st.markdown("""
<style>
.hero{padding:28px 32px;background:linear-gradient(135deg,#0a6e2c,#3aa860);
      color:#fff;border-radius:14px;margin-bottom:20px;
      box-shadow:0 4px 16px rgba(10,110,44,0.18)}
.hero h1{margin:0;font-size:34px;font-weight:700;letter-spacing:-0.5px}
.hero .tagline{font-size:17px;margin-top:8px;color:rgba(255,255,255,0.95)}
.hero .meta{font-size:13px;margin-top:10px;color:rgba(255,255,255,0.75)}

.stage-card{padding:12px 16px;border-radius:8px;margin:6px 0;
            border-left:4px solid #ddd;background:#fafafa;font-family:-apple-system,sans-serif}
.stage-done{border-left-color:#0a6e2c;background:#f4faf6}
.stage-running{border-left-color:#d4ad28;background:#fffdf3}
.stage-pending{border-left-color:#ddd;background:#fafafa;color:#aaa}

.kpi{background:#fff;border:1px solid #e0e0e0;border-radius:8px;padding:14px;text-align:center}
.kpi .l{font-size:11px;color:#888;text-transform:uppercase;letter-spacing:0.5px}
.kpi .v{font-size:28px;font-weight:700;color:#0a6e2c;margin:6px 0}
</style>
""", unsafe_allow_html=True)


# ── Pipeline boot (cached across the session) ─────────────────────────────
@st.cache_resource(show_spinner="Booting pipeline (one-time)...")
def boot_pipeline():
    from dotenv import load_dotenv
    load_dotenv(dotenv_path=PROJECT_ROOT / ".env")
    if not os.environ.get("GROQ_API_KEY"):
        return None, "GROQ_API_KEY non trovata in .env"
    nb_data = json.loads((APP_DIR / "Main.ipynb").read_text())
    glb = {"__builtins__": __builtins__, "display": lambda x: None,
           "__file__": str(APP_DIR / "Main.ipynb")}
    os.chdir(APP_DIR)
    for i, c in enumerate(nb_data["cells"]):
        if c["cell_type"] != "code": continue
        raw = "".join(c["source"])
        first_line = raw.split("\n", 1)[0]
        if raw.startswith("%%writefile") or first_line.startswith("# Phase 8"):
            continue
        src = "\n".join(l for l in raw.split("\n") if not l.strip().startswith(("!","%")))
        try: exec(src, glb, glb)
        except Exception as e:
            return None, f"boot error in cell {i}: {type(e).__name__}: {e}"
    return glb, None


def gauge(value, label="Reliability", height=260):
    color = "#0a6e2c" if value >= 70 else ("#d4ad28" if value >= 40 else "#c0392b")
    fig = go.Figure(go.Indicator(
        mode="gauge+number", value=value,
        number={"suffix":"/100", "font":{"size":40}},
        gauge={"axis":{"range":[0,100],"tickwidth":1},
               "bar":{"color":color,"thickness":0.75},
               "steps":[{"range":[0,40],"color":"#fff0ee"},
                        {"range":[40,70],"color":"#fff7d6"},
                        {"range":[70,100],"color":"#e8f5ed"}]},
        title={"text":label, "font":{"size":14, "color":"#666"}}))
    fig.update_layout(height=height, margin=dict(l=20,r=20,t=40,b=20))
    return fig


def radar(sub_scores, height=320):
    labels = list(sub_scores.keys())
    values = [sub_scores[k] for k in labels]
    fig = go.Figure(go.Scatterpolar(r=values, theta=labels, fill="toself",
                                     marker=dict(color="#0a6e2c"),
                                     line=dict(color="#0a6e2c")))
    fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0,100])),
                      showlegend=False, height=height,
                      margin=dict(l=40,r=40,t=20,b=20))
    return fig


def severity_bar(severity_breakdown, height=200):
    labels = ["critical","high","medium","low"]
    colors = ["#c0392b","#e67e22","#d4ad28","#95a5a6"]
    values = [severity_breakdown.get(l, 0) for l in labels]
    fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors, text=values, textposition="outside"))
    fig.update_layout(height=height, margin=dict(l=20,r=20,t=20,b=20),
                      yaxis=dict(showgrid=False), showlegend=False)
    return fig


# ── Sidebar ───────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("### 📥 Input")
    use_demo = st.toggle("Try with NoiPA `spesa` (demo)", value=False)
    uploaded = None if use_demo else st.file_uploader("Upload a CSV", type=["csv"])
    with st.expander("⚙️ Settings"):
        max_rows = st.number_input("Max rows", 200, 50000, 5000, step=500)
        show_narrative = st.checkbox("LLM-generated executive summary", value=False,
                                      help="Adds 1 short LLM call (~150 tokens) to write the report's summary in Italian.")
    st.markdown("---")
    st.markdown("**Group 17** · LUISS ML 2025/26")
    st.markdown("Captain ID `819621` · Reply *Agents for Data Quality*")
    mode = st.radio("Mode", ["🚀 Run pipeline", "🧪 Benchmark"], index=0)

# ── Hero ──────────────────────────────────────────────────────────────────
st.markdown("""
<div class="hero">
  <h1>🛠️ Agents for Data Quality</h1>
  <div class="tagline">A multi-agent LLM pipeline that validates, scores, and remediates any CSV — in seconds.</div>
  <div class="meta">LangGraph · Groq Llama 3.3 70B · 9 nodes (4 LLM agents + supervisor) · ~3-5k tokens / dataset</div>
</div>
""", unsafe_allow_html=True)


# ─────────────────────────── Mode 1: Run pipeline ─────────────────────────
if mode == "🚀 Run pipeline":
    if uploaded is None and not use_demo:
        # Welcome / explainer
        c1, c2 = st.columns([3, 2])
        with c1:
            st.markdown("### How it works")
            st.markdown("""
**6 specialized agents collaborate to assess your data:**

| Stage | Type | What it does |
|---|---|---|
| 🔌 Ingest + Discover | deterministic | Loads the CSV, infers schema rules from a sample |
| 🔍 Audit | deterministic | Runs 9 quality tools, accumulates issues |
| 🔧 Schema · Completeness · Consistency · Anomaly | **4 LLM agents** | Each plans fixes for its dimension, with column context |
| ⚡ Remediation | deterministic | Applies the merged plan with atomic tools |
| 🎯 Supervisor | deterministic | Aggregates 5 sub-scores → reliability 0-100 (ISO-8000) |
            """)
            if (IMAGES_DIR / "architecture_flowchart.png").exists():
                st.image(str(IMAGES_DIR / "architecture_flowchart.png"))
        with c2:
            st.markdown("### Quick start")
            st.markdown("👈 Upload a CSV in the sidebar, **or** flip the toggle to try with a demo dataset.")
            st.info("Pre-validated on 4 NoiPA datasets (Reply test fixtures). The pipeline is **schema-agnostic** — works on any tabular CSV.")
            st.markdown("### Benchmark — deterministic layer")
            ev = BENCHMARK_DIR / "evaluation_results.json"
            if ev.exists():
                r = json.loads(ev.read_text())
                k1, k2, k3 = st.columns(3)
                k1.metric("Precision", f"{r['Precision']:.2f}")
                k2.metric("Recall", f"{r['Recall']:.2f}")
                k3.metric("F1", f"{r['F1']:.2f}")
                st.caption(f"3 error types × 4 datasets, n_each=3")
        st.stop()

    # ── Got a CSV: load + preview
    if uploaded is not None:
        df_user = pd.read_csv(uploaded).head(max_rows)
        ds_name = Path(uploaded.name).stem
    else:
        df_user = pd.read_csv(DATA_DIR / "project_data_quality" / "spesa.csv").head(max_rows)
        ds_name = "spesa"

    st.subheader(f"📊 `{ds_name}`")
    cols = st.columns(4)
    cols[0].metric("Rows", f"{df_user.shape[0]:,}")
    cols[1].metric("Columns", df_user.shape[1])
    cols[2].metric("Numeric cols", df_user.select_dtypes(include="number").shape[1])
    cols[3].metric("String cols", df_user.select_dtypes(include="object").shape[1])
    with st.expander("Preview (first 10 rows)", expanded=False):
        st.dataframe(df_user.head(10), use_container_width=True, hide_index=True)

    if not st.button("🚀 Run multi-agent pipeline", type="primary", use_container_width=True):
        st.stop()

    glb, err = boot_pipeline()
    if err: st.error(err); st.stop()
    quality_graph = glb["quality_graph"]
    glb["datasets"][ds_name] = df_user

    # ── Live timeline of the 9 nodes via graph.stream() ───────────────────
    PIPELINE_STAGES = ["ingest", "discover", "audit", "schema",
                        "completeness", "consistency", "anomaly",
                        "remediation", "supervisor"]
    STAGE_LABELS = {
        "ingest":       "🔌 Ingest — loading dataframe",
        "discover":     "🔬 Discover — inferring schema rules",
        "audit":        "🔍 Audit — 9 deterministic tools running",
        "schema":       "🤖 Schema agent (LLM) — validity dimension",
        "completeness": "🤖 Completeness agent (LLM) — completeness dimension",
        "consistency":  "🤖 Consistency agent (LLM) — consistency + uniqueness",
        "anomaly":      "🤖 Anomaly agent (LLM) — accuracy dimension",
        "remediation":  "⚡ Remediation — applying fixes deterministically",
        "supervisor":   "🎯 Supervisor — aggregating reliability score",
    }

    timeline_box = st.container()
    placeholders = {s: timeline_box.empty() for s in PIPELINE_STAGES}
    for s in PIPELINE_STAGES:
        placeholders[s].markdown(f'<div class="stage-card stage-pending">⏸️ {STAGE_LABELS[s]}</div>', unsafe_allow_html=True)

    # Stream the graph and update the UI on every node completion.
    # graph.stream() emits only node-update events, NOT the initial input — so we
    # seed `state` with dataset_name and raw_df ourselves.
    state = {"dataset_name": ds_name, "raw_df": df_user.copy()}
    t0 = time.time()
    try:
        for event in quality_graph.stream({"dataset_name": ds_name, "raw_df": df_user.copy()}):
            for node_name, update in event.items():
                # Merge update into accumulated state (mirrors LangGraph reducers)
                for k, v in update.items():
                    if isinstance(state.get(k), list) and isinstance(v, list):
                        state[k] = state[k] + v
                    elif isinstance(state.get(k), dict) and isinstance(v, dict):
                        state[k] = {**state[k], **v}
                    else:
                        state[k] = v
                # Render done state for this node
                tag = state.get("audit_trail", [""])[-1] if state.get("audit_trail") else ""
                elapsed = time.time() - t0
                if node_name in placeholders:
                    placeholders[node_name].markdown(
                        f'<div class="stage-card stage-done">✅ <strong>{STAGE_LABELS[node_name]}</strong> · {elapsed:.1f}s<br>'
                        f'<span style="color:#666;font-size:13px">{tag}</span></div>',
                        unsafe_allow_html=True)
    except Exception as e:
        st.exception(e); st.stop()

    if show_narrative:
        with st.spinner("Generating narrative summary..."):
            state = glb["add_llm_narrative"](state)

    # Ensure raw_df present for the renderer
    if "raw_df" not in state: state["raw_df"] = df_user.copy()
    html = glb["render_quality_report"](state)

    st.success(f"Pipeline complete in {time.time()-t0:.1f}s · {len(state.get('issues',[]))} issues · {sum(1 for e in state.get('correction_log',[]) if e.get('applied'))} corrections applied.")

    # ── Results dashboard ────────────────────────────────────────────────
    score = state["reliability_score"]
    sub = state["sub_scores"]
    n_issues = len(state.get("issues", []))
    n_applied = sum(1 for e in state.get("correction_log", []) if e.get("applied"))

    g1, g2 = st.columns([1, 1])
    g1.plotly_chart(gauge(score, "Reliability score"), use_container_width=True)
    g2.plotly_chart(radar(sub, height=260), use_container_width=True)

    k = st.columns(5)
    for i, (lbl, val) in enumerate(sub.items()):
        k[i].markdown(f'<div class="kpi"><div class="l">{lbl}</div><div class="v">{val:.0f}</div></div>',
                      unsafe_allow_html=True)

    tab1, tab2, tab3, tab4, tab5 = st.tabs(["📋 Report", "🔍 Issues", "🔧 Corrections", "📁 Fixed CSV", "🛤️ Audit trail"])

    with tab1:
        st.components.v1.html(html, height=900, scrolling=True)
        st.download_button("⬇️ Download HTML report", html.encode(),
                           file_name=f"report_{ds_name}.html", mime="text/html")

    with tab2:
        sev = state.get("severity_breakdown", {})
        c1, c2 = st.columns([1, 2])
        c1.plotly_chart(severity_bar(sev), use_container_width=True)
        if n_issues > 0:
            issues_df = pd.DataFrame([{
                "tool": i["tool"], "type": i["issue_type"], "severity": i["severity"],
                "columns": ", ".join(i.get("columns", []) or []),
                "rows": i.get("row_count"), "message": (i.get("message") or "")[:140]
            } for i in state["issues"]])
            c2.dataframe(issues_df, use_container_width=True, hide_index=True, height=400)
        else:
            c2.success("No issues — clean dataset.")

    with tab3:
        log = state.get("correction_log", [])
        if log:
            log_df = pd.DataFrame(log)
            st.dataframe(log_df, use_container_width=True, hide_index=True, height=420)
        else:
            st.info("Nothing to correct.")

    with tab4:
        fixed_df = state.get("fixed_df", df_user)
        st.dataframe(fixed_df.head(50), use_container_width=True, hide_index=True)
        st.download_button("⬇️ Download fixed CSV",
                           fixed_df.to_csv(index=False).encode(),
                           file_name=f"fixed_{ds_name}.csv", mime="text/csv")

    with tab5:
        st.code("\n".join(state.get("audit_trail", [])), language=None)
        if state.get("errors"):
            with st.expander("Warnings", expanded=False):
                st.warning("\n".join(state["errors"]))


# ─────────────────────────── Mode 2: Benchmark ────────────────────────────
elif mode == "🧪 Benchmark":
    st.subheader("Phase 4 — Synthetic Benchmark")
    st.caption("Validation of the deterministic Phase 3 layer on 3 representative error types × 4 NoiPA datasets, n_each=3.")
    ev = BENCHMARK_DIR / "evaluation_results.json"
    if not ev.exists():
        st.error(f"Missing: {ev}. Run the notebook (Phase 4) to generate it."); st.stop()
    r = json.loads(ev.read_text())
    cols = st.columns(3)
    cols[0].plotly_chart(gauge(r["Precision"]*100, "Precision"), use_container_width=True)
    cols[1].plotly_chart(gauge(r["Recall"]*100, "Recall"), use_container_width=True)
    cols[2].plotly_chart(gauge(r["F1"]*100, "F1"), use_container_width=True)
    st.caption(f"TP={r['TP']} · FP={r['FP']} · FN={r['FN']}")

    st.markdown("---")
    if (IMAGES_DIR / "architecture_flowchart.png").exists():
        st.image(str(IMAGES_DIR / "architecture_flowchart.png"), caption="Pipeline architecture")
    if (IMAGES_DIR / "detection_heatmap.png").exists():
        st.image(str(IMAGES_DIR / "detection_heatmap.png"), caption="Benchmark metrics")


## Phase 8 — Smoke test (single-dataset verification)

Eseguiamo la pipeline su `df_spesa` (uno dei dataset NoiPA usati come *test fixture*) per confermare che il grafo gira end-to-end e produce uno score plausibile + report HTML. Niente di pre-computato viene committato in repo: questa cella è solo verifica.


In [ ]:
# Phase 8 — smoke test on one dataset (no artifacts persisted to outputs/)
print("Running smoke test on `spesa`...")
final_state, html = run_quality_pipeline(spesa, name="spesa", with_narrative=False)

print(f"Reliability: {final_state['reliability_score']}/100")
print(f"Sub-scores:  {final_state['sub_scores']}")
print(f"Issues:      {len(final_state['issues'])}")
print(f"Applied:     {sum(1 for e in final_state['correction_log'] if e.get('applied'))}/{len(final_state['correction_log'])}")
print(f"\nAudit trail:")
for line in final_state.get("audit_trail", []):
    print(f"  {line}")

print(f"\nHTML report length: {len(html):,} bytes (use Streamlit demo for interactive view)")
